<a href="https://colab.research.google.com/github/pullz6/Apache_Spark_Tryouts_V1/blob/main/Apache_Spark_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the original script used to get beginner understanding of Pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, udf

spark = SparkSession.builder.appName("demo").getOrCreate()

In [ ]:
df = spark.createDataFrame(
    [
        ("sue", 32),
        ("li", 3),
        ("bob", 75),
        ("heo", 13),
    ],
    ["first_name", "age"],
)

In [ ]:
df.show()

+----------+---+
|first_name|age|
+----------+---+
|       sue| 32|
|        li|  3|
|       bob| 75|
|       heo| 13|
+----------+---+



How to add a new column in spark

In [ ]:
df = df.withColumn(
    'life_stage',
     when(col("age") < 13, "child")
    .when(col("age").between(13, 19), "teenager")
    .otherwise("adult"),
)

In [ ]:
df.show()

+----------+---+----------+
|first_name|age|life_stage|
+----------+---+----------+
|       sue| 32|     adult|
|        li|  3|     child|
|       bob| 75|     adult|
|       heo| 13|  teenager|
+----------+---+----------+



Filtering for data

In [ ]:
df.where(col('life_stage').isin(['child','teenager'])).show()

+----------+---+----------+
|first_name|age|life_stage|
+----------+---+----------+
|        li|  3|     child|
|       heo| 13|  teenager|
+----------+---+----------+



Aggregation of dataframe

In [ ]:
df.select(avg(col('age'))).show()

+--------+
|avg(age)|
+--------+
|   30.75|
+--------+



df.groupby('life_stage') creates a GroupedData object.
You call .avg() directly on that object.

With no column specified, PySpark automatically applies the avg calculation to every numeric column in the original DataFrame for each group

In [ ]:
df.groupby('life_stage').avg().show()

+----------+--------+
|life_stage|avg(age)|
+----------+--------+
|     adult|    53.5|
|     child|     3.0|
|  teenager|    13.0|
+----------+--------+



You can also do it as

In [ ]:
df.groupby('life_stage').agg(avg(col('age'))).show()

+----------+--------+
|life_stage|avg(age)|
+----------+--------+
|     adult|    53.5|
|     child|     3.0|
|  teenager|    13.0|
+----------+--------+



Query the DataFrame with SQL

In [ ]:
spark.sql("select avg(age) from {df1}", df1=df).show()

+--------+
|avg(age)|
+--------+
|   30.75|
+--------+



In [ ]:
spark.sql("select life_stage,avg(age) from {df1} group by life_stage",df1=df ).show()

+----------+--------+
|life_stage|avg(age)|
+----------+--------+
|     adult|    53.5|
|     child|     3.0|
|  teenager|    13.0|
+----------+--------+



In [ ]:
df.show()

+----------+---+----------+
|first_name|age|life_stage|
+----------+---+----------+
|       sue| 32|     adult|
|        li|  3|     child|
|       bob| 75|     adult|
|       heo| 13|  teenager|
+----------+---+----------+



In [ ]:
def add_5_years(age):
  return age + 5

In [ ]:
add_5_years_udf = udf(add_5_years)

In [ ]:
df.withColumn('age_in_5_years',add_5_years_udf(col('age'))).show()

+----------+---+----------+--------------+
|first_name|age|life_stage|age_in_5_years|
+----------+---+----------+--------------+
|       sue| 32|     adult|            37|
|        li|  3|     child|             8|
|       bob| 75|     adult|            80|
|       heo| 13|  teenager|            18|
+----------+---+----------+--------------+



In [ ]:
df.rdd()